In [121]:
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict

In [122]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
FANTOM = DATA / "fantom_cat"
PREPROCESSING = Path("../preprocessing")
IPS = DATA_INITIAL / "deg/rnaseq_res_ips"
FIBRO = DATA_INITIAL / "deg/rnaseq_res_fibro"
IPS_INDEX = PREPROCESSING / "deg_index/ips_deg_index"
FIBRO_INDEX = PREPROCESSING / "deg_index/fibro_deg_index"
BIOMART = DATA / "biomart"
BIOMART_INITIAL = DATA_INITIAL / "biomart"
DATA_INITIAL = Path("../data_initial").resolve()
GENCODE = DATA / "gencode"
GENCODE_INITIAL = DATA_INITIAL / "gencode"
IPS_PREPROC = PREPROCESSING / "deg/fibro"
FIBRO_PREPROC = PREPROCESSING / "deg/ips"

In [123]:
def deg_update(deg_expr_filt_columns_sbst):
    columns_1 = ["DEG.ens_gene","log2FC","pvalue","fdr","ZScore","target.aso","target.ens_gene","up_or_down_expr"]
    columns_2 = ["ens_gene", "new-ens_gene","new-gene_name"]
    deg_expr_filt_columns_sbst_gencode47 = pd.merge(left=deg_expr_filt_columns_sbst[columns_1], right=fantom_l2_genes_update_with_names[columns_2], how="inner", left_on="target.ens_gene", right_on="ens_gene")
    del deg_expr_filt_columns_sbst_gencode47["ens_gene"]
    del deg_expr_filt_columns_sbst_gencode47["target.ens_gene"]
    deg_expr_filt_columns_sbst_gencode47 = deg_expr_filt_columns_sbst_gencode47.rename(columns={"new-ens_gene":"target.new-ens_gene", "new-gene_name":"target.new-gene_name"})
    columns_1 = ["DEG.ens_gene","log2FC","pvalue","fdr","ZScore","target.aso","target.new-ens_gene","target.new-gene_name","up_or_down_expr"]
    columns_2 = ["ens_gene", "new-ens_gene","new-gene_name"]
    deg_expr_filt_columns_sbst_gencode47 = pd.merge(left=deg_expr_filt_columns_sbst_gencode47[columns_1], right=fantom_l2_genes_update_with_names[columns_2], how="inner", left_on="DEG.ens_gene", right_on="ens_gene")
    del deg_expr_filt_columns_sbst_gencode47["ens_gene"]
    del deg_expr_filt_columns_sbst_gencode47["DEG.ens_gene"]
    deg_expr_filt_columns_sbst_gencode47 = deg_expr_filt_columns_sbst_gencode47.rename(columns={"new-ens_gene":"DEG.new-ens_gene", "new-gene_name":"DEG.new-gene_name"})
    columns = ["DEG.new-ens_gene","DEG.new-gene_name","log2FC","pvalue","fdr","ZScore","target.aso","target.new-ens_gene","target.new-gene_name","up_or_down_expr"]
    deg_expr_filt_columns_sbst_gencode47 = deg_expr_filt_columns_sbst_gencode47[columns]
    mask_1 = deg_expr_filt_columns_sbst_gencode47["target.new-ens_gene"].apply(lambda x: "retired" not in x)
    mask_2 = deg_expr_filt_columns_sbst_gencode47["DEG.new-ens_gene"].apply(lambda x: "retired" not in x)
    deg_expr_filt_columns_sbst_gencode47_without_retired = deg_expr_filt_columns_sbst_gencode47[mask_1 & mask_2]
    return deg_expr_filt_columns_sbst_gencode47_without_retired

In [124]:
to_ips = IPS / "EdgeR.table.all.target.tsv.gz"
ips_expr = pd.read_csv(to_ips, sep="\t")
ips_expr.shape

(3445156, 10)

In [125]:
ips_expr.head(1)

,DEG.geneID,DEG.HGNC,EdgeR.log2FC,EdgeR.log2CPM,EdgeR.pvalue,EdgeR.FDR,EdgeR.Zscore,target.gapmerID,target.geneID,target.HGNC
0,CATG00000000155,CATG00000000155.1,0.257141,3.591144,0.550588,1.0,0.620927,G0234787_01,ENSG00000234787,LINC00458


In [129]:
ips_expr["target.geneID"].nunique()

123

In [130]:
ips_expr["target.gapmerID"].nunique()

317

In [131]:
ips_expr_filt = ips_expr.query("`EdgeR.FDR` < 0.05 & abs(`EdgeR.Zscore`) > 1.645 & abs(`EdgeR.log2FC`) > 0.5")
ips_expr_filt.reset_index(inplace=True, drop=True)
ips_expr_filt.shape

(15629, 10)

In [132]:
ips_expr_filt.head(1)

,DEG.geneID,DEG.HGNC,EdgeR.log2FC,EdgeR.log2CPM,EdgeR.pvalue,EdgeR.FDR,EdgeR.Zscore,target.gapmerID,target.geneID,target.HGNC
0,ENSG00000205531,NAP1L4,-0.691972,7.349218,0.000004,0.042455,-5.061144,G0234787_08,ENSG00000234787,LINC00458


In [133]:
ips_expr_filt["up_or_down_expr"] = np.where(ips_expr_filt["EdgeR.log2FC"] > 0, "up", "down")

/tmp/ipykernel_115216/601272801.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  ips_expr_filt["up_or_down_expr"] = np.where(ips_expr_filt["EdgeR.log2FC"] > 0, "up", "down")


In [134]:
ips_expr_filt["DEG.geneID"].nunique()

5534

In [135]:
ips_expr_filt["target.geneID"].nunique()

119

In [136]:
ips_expr_filt["target.gapmerID"].nunique()

259

In [137]:
to_fibro = FIBRO / "DESeq2_genes_ASO_all.tsv.gz"
fibro_expr = pd.read_csv(to_fibro, sep="\t")
fibro_expr.shape

(5748772, 13)

In [138]:
fibro_expr.head(1)

,Description,ASO_CAGE,KD.geneSymbol,perturb_id,geneID,geneSymbol,baseMean,log2FC,pvalue,fdr,KD.tpm.ave,NC.tpm.ave,ZScore
0,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000200169,RNU5D-1,134.656674,-2.325717,1.640034e-11,2.548449e-07,3.026571,15.427402,-3.106205


In [139]:
fibro_expr["KD.geneSymbol"].nunique()

169

In [140]:
fibro_expr["perturb_id"].nunique()

370

In [141]:
fibro_expr_filt = fibro_expr.query("fdr < 0.05 & abs(ZScore) > 1.645 & abs(log2FC) > 0.5")
ips_expr_filt.reset_index(inplace=True, drop=True)
fibro_expr_filt.shape

(145540, 13)

In [142]:
fibro_expr_filt.head(1)

,Description,ASO_CAGE,KD.geneSymbol,perturb_id,geneID,geneSymbol,baseMean,log2FC,pvalue,fdr,KD.tpm.ave,NC.tpm.ave,ZScore
0,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000200169,RNU5D-1,134.656674,-2.325717,1.640034e-11,2.548449e-07,3.026571,15.427402,-3.106205


In [143]:
fibro_expr_filt["up_or_down_expr"] = np.where(fibro_expr_filt["log2FC"] > 0, "up", "down")

/tmp/ipykernel_115216/515906159.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fibro_expr_filt["up_or_down_expr"] = np.where(fibro_expr_filt["log2FC"] > 0, "up", "down")


В данной таблице нет названия ENSG для таргентных нокдаунных lncRNA. Добавим их из таблицы: ASO_names.info

In [144]:
to_fibro_aso = FIBRO / "ASO_names.info"
fibro_aso = pd.read_csv(to_fibro_aso, sep="\t")
fibro_aso.head(3)

,aso_id,aso_id_short,prmtrID,prmtrName,trnscptID,trnscptName,trnscptType,geneID,geneName,geneType
0,ASO_C0000830_02,C0000830,CUFT00000000830.T0,chr1_50977446_50977745_+,p012@CDKN2C,CUFT00000000830.T0,__na,ENSG00000123080,CDKN2C,protein_coding
1,ASO_C0000830_08,C0000830,CUFT00000000830.T0,chr1_50977446_50977745_+,p012@CDKN2C,CUFT00000000830.T0,__na,ENSG00000123080,CDKN2C,protein_coding
2,ASO_C0008202_01,C0008202,CUFT00000008202.T0,chr4_183350680_183350979_-,p003@CLDN22,CUFT00000008202.T0,__na,ENSG00000177300,CLDN22,protein_coding


In [145]:
fibro_aso.shape

(350, 10)

In [146]:
fibro_expr_filt = pd.merge(left=fibro_expr_filt, left_on="perturb_id",
         right=fibro_aso[["aso_id", "geneID"]], right_on="aso_id")
fibro_expr_filt.head()

,Description,ASO_CAGE,KD.geneSymbol,perturb_id,geneID_x,geneSymbol,baseMean,log2FC,pvalue,fdr,KD.tpm.ave,NC.tpm.ave,ZScore,up_or_down_expr,aso_id,geneID_y
0,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000200169,RNU5D-1,134.656674,-2.325717,1.640034e-11,2.548449e-07,3.026571,15.427402,-3.106205,down,ASO_G0268895_01,ENSG00000268895
1,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000138675,FGF5,1788.727654,-1.371879,7.814666e-11,6.071605e-07,75.054256,196.439355,-2.756217,down,ASO_G0268895_01,ENSG00000268895
2,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000252521,RNU5D-2P,108.388065,-1.925674,1.102746e-09,5.711854e-06,3.178508,12.249495,-2.534245,down,ASO_G0268895_01,ENSG00000268895
3,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,ENSG00000268895,A1BG-AS1,100.134784,-1.542280,3.509193e-09,1.363234e-05,3.802250,11.079833,-7.161737,down,ASO_G0268895_01,ENSG00000268895
4,Targeted lncRNA,2,A1BG-AS1,ASO_G0268895_01,CATG00000098831,CATG00000098831.1,163.640506,1.153985,1.174760e-08,3.480974e-05,27.770456,12.658186,2.301412,up,ASO_G0268895_01,ENSG00000268895


In [147]:
fibro_expr_filt[fibro_expr_filt["geneID_y"] == "ENSG00000268895"]["perturb_id"].nunique()

2

Перед обновлением идентификаторов приведем таблицы для fibro и для ips к общим полям:

In [148]:
ips_columns = ["DEG.geneID","DEG.HGNC","EdgeR.log2FC","EdgeR.pvalue","EdgeR.FDR","EdgeR.Zscore","target.gapmerID","target.geneID","target.HGNC","up_or_down_expr"]

In [149]:
len(ips_columns)

10

In [150]:
fibro_columns = ["geneID_x","geneSymbol","log2FC","pvalue","fdr","ZScore","perturb_id","geneID_y","KD.geneSymbol","up_or_down_expr"]

In [151]:
len(fibro_columns)

10

In [152]:
correct_columns = ["DEG.ens_gene","DEG.gene_name","log2FC","pvalue","fdr","ZScore","target.aso","target.ens_gene","target.gene_name","up_or_down_expr"]

In [153]:
fibro_expr_filt_columns_sbst = fibro_expr_filt[fibro_columns].rename(columns={i:j for i,j in zip(fibro_columns, correct_columns)})
fibro_expr_filt_columns_sbst.head(1)

,DEG.ens_gene,DEG.gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.ens_gene,target.gene_name,up_or_down_expr
0,ENSG00000200169,RNU5D-1,-2.325717,1.640034e-11,2.548449e-07,-3.106205,ASO_G0268895_01,ENSG00000268895,A1BG-AS1,down


In [154]:
ips_expr_filt_columns_sbst = ips_expr_filt[ips_columns].rename(columns={i:j for i,j in zip(ips_columns, correct_columns)})
ips_expr_filt_columns_sbst.head(1)

,DEG.ens_gene,DEG.gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.ens_gene,target.gene_name,up_or_down_expr
0,ENSG00000205531,NAP1L4,-0.691972,0.000004,0.042455,-5.061144,G0234787_08,ENSG00000234787,LINC00458,down


In [155]:
fibro_expr_filt_columns_sbst["target.aso"] = fibro_expr_filt_columns_sbst["target.aso"].apply(lambda x: x.split("ASO_")[1])

Приступим к обновлению: все гены из этих таблиц с большой долей вероятности взяты из аннотации fantom_l2_permissive, которая сделана на GRCh37. \
На предыдущем этапе в момент переаннотирования himorna мной была обновлена fantom_l2_permissive. Она была liftover на GRCh38, а также были обновлены устаревшие ensg* идентификаторы генов.

In [156]:
to_file = FANTOM / 'fantom_l2-genes_with_name-hg38-all_info.tsv'
fantom_l2_genes_update_with_names = pd.read_csv(to_file, sep="\t")
fantom_l2_genes_update_with_names.head(3)

,chrom,start,end,ens_gene,gene_name,score,strand,new-ens_gene,new-gene_name
0,chr1,159537,162485,CATG00000042730,CATG00000042730,.,-,CATG00000042730,CATG00000042730
1,chr1,273882,274416,CATG00000042731,CATG00000042731,.,-,CATG00000042731,CATG00000042731
2,chr1,588120,634563,CATG00000042732,CATG00000042732,.,-,CATG00000042732,CATG00000042732


In [157]:
fantom_l2_genes_update_with_names.shape

(123997, 9)

In [32]:
fantom_l2_genes_update_with_names[fantom_l2_genes_update_with_names["ens_gene"] == "ENSG00000107960"]

,chrom,start,end,ens_gene,score,strand,new-ens_gene,new-gene_name
15324,chr10,103856799,103918582,ENSG00000107960,.,-,ENSG00000107960,STN1


In [33]:
len(set(fibro_expr_filt_columns_sbst["DEG.ens_gene"].unique()) | set(fibro_expr_filt_columns_sbst["target.ens_gene"].unique()))

17041

In [34]:
len(set(ips_expr_filt_columns_sbst["DEG.ens_gene"].unique()) | set(ips_expr_filt_columns_sbst["target.ens_gene"].unique()))

5616

In [95]:
ips_expr_filt_columns_sbst_gencode47_without_retired = deg_update(deg_expr_filt_columns_sbst=ips_expr_filt_columns_sbst)
ips_expr_filt_columns_sbst_gencode47_without_retired

,DEG.new-ens_gene,DEG.new-gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.new-ens_gene,target.new-gene_name,up_or_down_expr
0,ENSG00000205531,NAP1L4,-0.691972,3.899614e-06,0.042455,-5.061144,G0234787_08,ENSG00000234787,LINC00458,down
1,ENSG00000089916,GPATCH2L,-1.041246,3.913478e-05,0.043336,-3.953852,R0057746_03,CATG00000070943,CATG00000070943,down
2,ENSG00000099194,SCD,-0.635000,2.054724e-07,0.000870,-2.709444,R0057746_03,CATG00000070943,CATG00000070943,down
3,ENSG00000113161,HMGCR,-0.912822,1.601495e-05,0.029059,-3.682696,R0057746_03,CATG00000070943,CATG00000070943,down
4,ENSG00000128829,EIF2AK4,-0.887877,2.397361e-07,0.000870,-4.709886,R0057746_03,CATG00000070943,CATG00000070943,down
...,...,...,...,...,...,...,...,...,...,...
15336,ENSG00000259132,ENSG00000259132,0.951549,3.207729e-05,0.002283,3.083371,G0230844_AD_07,ENSG00000230844,ZNF674-AS1,up
15337,ENSG00000259203,ENSG00000259203,1.364151,6.251776e-05,0.003601,1.693698,G0230844_AD_07,ENSG00000230844,ZNF674-AS1,up
15338,ENSG00000262481,TMEM256-PLSCR3,-0.742945,4.097390e-04,0.013477,-2.084815,G0230844_AD_07,ENSG00000230844,ZNF674-AS1,down
15340,ENSG00000265565,MIR3143,-0.673589,3.604504e-05,0.002453,-3.019324,G0230844_AD_07,ENSG00000230844,ZNF674-AS1,down


In [107]:
ips_expr_filt_columns_sbst_gencode47_without_retired[["DEG.new-ens_gene","target.new-ens_gene","DEG.new-gene_name","target.new-gene_name"]].nunique()

DEG.new-ens_gene        5359
target.new-ens_gene      114
DEG.new-gene_name       5357
target.new-gene_name     114
dtype: int64

In [99]:
ips_expr_filt_columns_sbst["target.ens_gene"].nunique()

119

In [100]:
ips_expr_filt_columns_sbst["DEG.ens_gene"].nunique()

5534

In [96]:
fibro_expr_filt_columns_sbst_gencode47_without_retired = deg_update(deg_expr_filt_columns_sbst=fibro_expr_filt_columns_sbst)
fibro_expr_filt_columns_sbst_gencode47_without_retired

,DEG.new-ens_gene,DEG.new-gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.new-ens_gene,target.new-gene_name,up_or_down_expr
0,ENSG00000200169,RNU5D-1,-2.325717,1.640034e-11,2.548449e-07,-3.106205,G0268895_01,ENSG00000268895,A1BG-AS1,down
1,ENSG00000138675,FGF5,-1.371879,7.814666e-11,6.071605e-07,-2.756217,G0268895_01,ENSG00000268895,A1BG-AS1,down
2,ENSG00000252521,RNU5D-2P,-1.925674,1.102746e-09,5.711854e-06,-2.534245,G0268895_01,ENSG00000268895,A1BG-AS1,down
3,ENSG00000268895,A1BG-AS1,-1.542280,3.509193e-09,1.363234e-05,-7.161737,G0268895_01,ENSG00000268895,A1BG-AS1,down
4,CATG00000098831,CATG00000098831,1.153985,1.174760e-08,3.480974e-05,2.301412,G0268895_01,ENSG00000268895,A1BG-AS1,up
...,...,...,...,...,...,...,...,...,...,...
141356,ENSG00000232187,FTH1P7,2.554486,3.241567e-05,4.609803e-02,5.237118,G0267886_AD_04,ENSG00000183850,ZNF730,up
141357,ENSG00000142961,MOB3C,0.749422,6.254070e-06,4.815008e-02,3.582150,G0269293_01,ENSG00000269293,ZSCAN16-AS1,up
141358,ENSG00000132613,MTSS2,-0.535628,1.170372e-06,1.853869e-02,-1.732862,G0269293_AD_10,ENSG00000269293,ZSCAN16-AS1,down
141359,ENSG00000264395,MIR3193,1.530984,4.610931e-06,3.651857e-02,2.419552,G0269293_AD_10,ENSG00000269293,ZSCAN16-AS1,up


In [106]:
fibro_expr_filt_columns_sbst_gencode47_without_retired[["DEG.new-ens_gene","target.new-ens_gene","DEG.new-gene_name","target.new-gene_name"]].nunique()

DEG.new-ens_gene        16340
target.new-ens_gene       148
DEG.new-gene_name       16328
target.new-gene_name      148
dtype: int64

In [103]:
fibro_expr_filt_columns_sbst["target.ens_gene"].nunique()

154

In [104]:
fibro_expr_filt_columns_sbst["DEG.ens_gene"].nunique()

17037

In [108]:
to_save = FIBRO_PREPROC / "fibro_df_with_deg_target_ens_genes_updated.tsv"
fibro_expr_filt_columns_sbst_gencode47_without_retired.to_csv(to_save, sep="\t", index=False)

In [110]:
to_save = IPS_PREPROC / "ips_df_with_deg_target_ens_genes_updated.tsv"
ips_expr_filt_columns_sbst_gencode47_without_retired.to_csv(to_save, sep="\t", index=False)

In [341]:
fibro_expr_unique_ens_gene = pd.DataFrame((set(fibro_expr_filt_columns_sbst["DEG.ens_gene"].unique()) | set(fibro_expr_filt_columns_sbst["target.ens_gene"].unique())), columns=["ens_gene"])
fibro_expr_unique_ens_gene

,ens_gene
0,ENSG00000162222
1,ENSG00000006747
2,ENSG00000103047
3,ENSG00000085719
4,ENSG00000151657
...,...
17036,ENSG00000075399
17037,CATG00000103770
17038,ENSG00000226752
17039,CATG00000000700


In [342]:
fibro_expr_unique_ens_gene_merged_with_fl2_gh38 = pd.merge(left=fibro_expr_unique_ens_gene, right=fantom_genes_hg38_all_info, how="left", left_on="ens_gene", right_on="name")
fibro_expr_unique_ens_gene_merged_with_fl2_gh38

,ens_gene,chrom,start,end,name,score,strand,actual_name,actual_gene_name
0,ENSG00000162222,chr11,62634955.0,62744383.0,ENSG00000162222,.,+,ENSG00000162222,TTC9C
1,ENSG00000006747,chr7,12567399.0,12667735.0,ENSG00000006747,.,+,ENSG00000006747,SCIN
2,ENSG00000103047,chr16,68843541.0,69109623.0,ENSG00000103047,.,+,ENSG00000103047,TANGO6
3,ENSG00000085719,chr8,86483844.0,86566900.0,ENSG00000085719,.,+,ENSG00000085719,CPNE3
4,ENSG00000151657,chr10,7749471.0,7787993.0,ENSG00000151657,.,-,ENSG00000151657,KIN
...,...,...,...,...,...,...,...,...,...
17036,ENSG00000075399,chr16,89705498.0,89723823.0,ENSG00000075399,.,-,ENSG00000075399,VPS9D1
17037,CATG00000103770,chr8,118070701.0,118076051.0,CATG00000103770,.,-,CATG00000103770,NaN
17038,ENSG00000226752,chr9,120843084.0,120860221.0,ENSG00000226752,.,+,ENSG00000226752,CUTALP
17039,CATG00000000700,chr10,103936687.0,103967025.0,CATG00000000700,.,-,CATG00000000700,NaN


In [343]:
fibro_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"] = fibro_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"].astype(str)

In [344]:
fibro_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"].apply(lambda x: x == "nan").sum()

62

In [345]:
mask = fibro_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"].apply(lambda x: "ENS" in x or "CATG" in x)
fibro_expr_unique_ens_gene_merged_with_fl2_gh38[mask]

,ens_gene,chrom,start,end,name,score,strand,actual_name,actual_gene_name
0,ENSG00000162222,chr11,62634955.0,62744383.0,ENSG00000162222,.,+,ENSG00000162222,TTC9C
1,ENSG00000006747,chr7,12567399.0,12667735.0,ENSG00000006747,.,+,ENSG00000006747,SCIN
2,ENSG00000103047,chr16,68843541.0,69109623.0,ENSG00000103047,.,+,ENSG00000103047,TANGO6
3,ENSG00000085719,chr8,86483844.0,86566900.0,ENSG00000085719,.,+,ENSG00000085719,CPNE3
4,ENSG00000151657,chr10,7749471.0,7787993.0,ENSG00000151657,.,-,ENSG00000151657,KIN
...,...,...,...,...,...,...,...,...,...
17036,ENSG00000075399,chr16,89705498.0,89723823.0,ENSG00000075399,.,-,ENSG00000075399,VPS9D1
17037,CATG00000103770,chr8,118070701.0,118076051.0,CATG00000103770,.,-,CATG00000103770,NaN
17038,ENSG00000226752,chr9,120843084.0,120860221.0,ENSG00000226752,.,+,ENSG00000226752,CUTALP
17039,CATG00000000700,chr10,103936687.0,103967025.0,CATG00000000700,.,-,CATG00000000700,NaN


In [346]:
mask = fibro_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"].apply(lambda x: "<" in x)
fibro_expr_unique_ens_gene_merged_with_fl2_gh38[mask]

,ens_gene,chrom,start,end,name,score,strand,actual_name,actual_gene_name
19,ENSG00000265381,chr1,160205384.0,160205464.0,ENSG00000265381,.,+,<retired>,NaN
112,ENSG00000233937,chr5,181244485.0,181272167.0,ENSG00000233937,.,+,<retired>,NaN
127,ENSG00000267545,chr19,45178784.0,45208862.0,ENSG00000267545,.,+,<retired>,NaN
193,ENSG00000256873,chr22,38214945.0,38224904.0,ENSG00000256873,.,-,<retired>,NaN
248,ENSG00000264908,chr13,76886321.0,76886405.0,ENSG00000264908,.,-,<retired>,NaN
...,...,...,...,...,...,...,...,...,...
16792,ENSG00000272635,chr19,27793464.0,27984984.0,ENSG00000272635,.,+,<retired>,NaN
16801,ENSG00000213971,chr19,23058906.0,23274219.0,ENSG00000213971,.,-,<retired>,NaN
16827,ENSG00000171471,chr12,116557706.0,116576620.0,ENSG00000171471,.,+,<retired>,NaN
16915,ENSG00000265575,chr2,131491160.0,131492099.0,ENSG00000265575,.,-,<retired>,NaN


fibro:\
Вывод: несмотря на то что авторы статей говорят, что использовали l2 permissive, мы обнаружили 62 гена, которые не входят в эту разметку. \
414 идентификатора устарели и не вошли в fantom l2 permissive на GRCh38.
Остлось 16565 генов

In [347]:
(17041 - 16565) / 17041  * 100

2.7932633061440058

In [348]:
17041 - 16565 - 62 - 414

0

In [349]:
ips_expr_unique_ens_gene = pd.DataFrame((set(ips_expr_filt_columns_sbst["DEG.ens_gene"].unique()) | set(ips_expr_filt_columns_sbst["target.ens_gene"].unique())), columns=["ens_gene"])
ips_expr_unique_ens_gene

,ens_gene
0,ENSG00000162222
1,ENSG00000256667
2,ENSG00000103047
3,ENSG00000198929
4,ENSG00000167595
...,...
5611,ENSG00000182500
5612,ENSG00000136153
5613,ENSG00000079805
5614,ENSG00000117569


In [350]:
ips_expr_unique_ens_gene_merged_with_fl2_gh38 = pd.merge(left=ips_expr_unique_ens_gene, right=fantom_genes_hg38_all_info, how="left", left_on="ens_gene", right_on="name")
ips_expr_unique_ens_gene_merged_with_fl2_gh38

,ens_gene,chrom,start,end,name,score,strand,actual_name,actual_gene_name
0,ENSG00000162222,chr11,62634955.0,62744383.0,ENSG00000162222,.,+,ENSG00000162222,TTC9C
1,ENSG00000256667,chr12,10579724.0,10613591.0,ENSG00000256667,.,-,ENSG00000256667,KLRA1P
2,ENSG00000103047,chr16,68843541.0,69109623.0,ENSG00000103047,.,+,ENSG00000103047,TANGO6
3,ENSG00000198929,chr1,162069740.0,162388044.0,ENSG00000198929,.,+,ENSG00000198929,NOS1AP
4,ENSG00000167595,chr19,35747246.0,35853097.0,ENSG00000167595,.,+,ENSG00000167595,PROSER3
...,...,...,...,...,...,...,...,...,...
5611,ENSG00000182500,chr12,121624976.0,121655075.0,ENSG00000182500,.,+,<retired>,NaN
5612,ENSG00000136153,chr13,75589497.0,75871030.0,ENSG00000136153,.,+,ENSG00000136153,LMO7
5613,ENSG00000079805,chr19,10718079.0,10833488.0,ENSG00000079805,.,+,ENSG00000079805,DNM2
5614,ENSG00000117569,chr1,96721788.0,96984971.0,ENSG00000117569,.,+,ENSG00000117569,PTBP2


In [351]:
ips_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"] = ips_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"].astype(str)

In [352]:
ips_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"].apply(lambda x: x == "nan").sum()

18

In [353]:
mask = ips_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"].apply(lambda x: "ENS" in x or "CATG" in x)
ips_expr_unique_ens_gene_merged_with_fl2_gh38[mask].shape

(5491, 9)

In [354]:
mask = ips_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"].apply(lambda x: "<" in x)
ips_expr_unique_ens_gene_merged_with_fl2_gh38[mask].shape

(107, 9)

ips: \
Вывод: несмотря на то что авторы статей говорят, что использовали l2 permissive, мы обнаружили 18 генов, которые не входят в эту разметку.
107 идентификатор устарел и не вошел в fantom l2 permissive на GRCh38. \
Остлось 5491 генов

In [355]:
(5616 - 5491) / 5616  * 100

2.2257834757834756

In [356]:
5616 - 5491 - 18 - 107

0

In [359]:
mask = fibro_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"].apply(lambda x: "ENS" in x or "CATG" in x)
fibro_good_genes_mapping_df = fibro_expr_unique_ens_gene_merged_with_fl2_gh38[mask][["ens_gene","actual_name", "actual_gene_name"]]

In [362]:
mask = ips_expr_unique_ens_gene_merged_with_fl2_gh38["actual_name"].apply(lambda x: "ENS" in x or "CATG" in x)
ips_good_genes_mapping_df = ips_expr_unique_ens_gene_merged_with_fl2_gh38[mask][["ens_gene","actual_name", "actual_gene_name"]]

In [365]:
def update_deg_and_target_ens_gene(exp_df, exp_fantom_mapping_df):
    exp_df_with_deg_ens_genes_updated = pd.merge(left=exp_df, right=exp_fantom_mapping_df, how="inner", left_on="DEG.ens_gene", right_on="ens_gene")
    exp_df_with_deg_ens_genes_updated = exp_df_with_deg_ens_genes_updated.rename(columns={"actual_name":"DEG.ens_gene-new"})
    exp_df_with_deg_target_ens_genes_updated = pd.merge(left=exp_df_with_deg_ens_genes_updated, right=exp_fantom_mapping_df, how="inner", left_on="target.ens_gene", right_on="ens_gene")
    exp_df_with_deg_target_ens_genes_updated = exp_df_with_deg_target_ens_genes_updated.rename(columns={"actual_name":"target.ens_gene-new"})
    columns = exp_df_with_deg_target_ens_genes_updated.columns.to_list()
    columns.remove("ens_gene_x")
    columns.remove("ens_gene_y")
    exp_df_with_deg_target_ens_genes_updated = exp_df_with_deg_target_ens_genes_updated[columns]
    exp_df_with_deg_target_ens_genes_updated = exp_df_with_deg_target_ens_genes_updated.rename(columns={"actual_gene_name_x":"DEG.gene_name-new", "actual_gene_name_y":"target.gene_name-new"})
    return exp_df_with_deg_target_ens_genes_updated

In [367]:
fibro_df_with_deg_target_ens_genes_updated = update_deg_and_target_ens_gene(exp_df=fibro_expr_filt_columns_sbst, exp_fantom_mapping_df=fibro_good_genes_mapping_df)
fibro_df_with_deg_target_ens_genes_updated.head(1)

,DEG.ens_gene,DEG.gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.ens_gene,target.gene_name,up_or_down_expr,DEG.ens_gene-new,DEG.gene_name-new,target.ens_gene-new,target.gene_name-new
0,ENSG00000200169,RNU5D-1,-2.325717,1.640034e-11,2.548449e-07,-3.106205,G0268895_01,ENSG00000268895,A1BG-AS1,down,ENSG00000200169,RNU5D-1,ENSG00000268895,A1BG-AS1


In [368]:
len(fibro_df_with_deg_target_ens_genes_updated)

128114

In [371]:
fibro_df_with_deg_target_ens_genes_updated[["DEG.ens_gene-new","target.ens_gene-new","DEG.gene_name-new","target.gene_name-new"]].nunique()

DEG.ens_gene-new        16340
target.ens_gene-new       148
DEG.gene_name-new       14933
target.gene_name-new      137
dtype: int64

In [372]:
ips_df_with_deg_target_ens_genes_updated = update_deg_and_target_ens_gene(exp_df=ips_expr_filt_columns_sbst, exp_fantom_mapping_df=ips_good_genes_mapping_df)
ips_df_with_deg_target_ens_genes_updated.head(1)

,DEG.ens_gene,DEG.gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.ens_gene,target.gene_name,up_or_down_expr,DEG.ens_gene-new,DEG.gene_name-new,target.ens_gene-new,target.gene_name-new
0,ENSG00000205531,NAP1L4,-0.691972,0.000004,0.042455,-5.061144,G0234787_08,ENSG00000234787,LINC00458,down,ENSG00000205531,NAP1L4,ENSG00000234787,LINC00458


In [373]:
len(ips_df_with_deg_target_ens_genes_updated)

14743

In [374]:
ips_df_with_deg_target_ens_genes_updated[["DEG.ens_gene-new","target.ens_gene-new","DEG.gene_name-new","target.gene_name-new"]].nunique()

DEG.ens_gene-new        5359
target.ens_gene-new      114
DEG.gene_name-new       5295
target.gene_name-new      96
dtype: int64

In [378]:
to_save = FIBRO_PREPROC / "fibro_df_with_deg_target_ens_genes_updated.tsv"
fibro_df_with_deg_target_ens_genes_updated.to_csv(to_save, sep="\t", index=False)

In [379]:
to_save = IPS_PREPROC / "ips_df_with_deg_target_ens_genes_updated.tsv"
ips_df_with_deg_target_ens_genes_updated.to_csv(to_save, sep="\t", index=False)